# 03 - Controlled classical models

This notebook compares word, character, and o200k BPE representations while holding the downstream Logistic Regression classifier and data splits fixed. The design isolates representation choice as the main experimental difference.

**Inputs:** Clean train/validation/test rows and the shared attacked-test corpus.  
**Outputs:** Selected classical models, validation search results, clean scores, and raw prediction partitions.

## Setup and prediction format

The setup contains only the loaders, tokenizer adapter, prediction routine, and artifact naming used below. Every prediction partition keeps the same columns—example identity, condition, gold label, prediction, gold-class probability, correctness, and number of text changes—so defenses and statistics can consume all model families uniformly.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import accuracy_score, f1_score

# Same path setup as notebook 00; repeated for independent execution.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CONFIG = yaml.safe_load((ROOT / 'config/experiment.yaml').read_text(encoding='utf-8'))

PATHS = {key: ROOT / value for key, value in CONFIG['paths'].items()}
for directory in (PATHS['models'], PATHS['predictions'], PATHS['tables'],):
    directory.mkdir(parents=True, exist_ok=True)

import tiktoken

_BPE_ENCODING = tiktoken.get_encoding('o200k_base')

def bpe_tokenize(text):
    return [f'tok_{token_id}' for token_id in _BPE_ENCODING.encode(text)]

def predict_sklearn(model, texts, labels, chunk_size=None):
    chunk_size = chunk_size or CONFIG['classical']['prediction_chunk_size']
    predictions = []
    probabilities = []
    labels_array = np.asarray(labels, dtype=int)
    classes = np.asarray(model.classes_, dtype=int)
    class_to_column = {label: idx for idx, label in enumerate(classes)}
    for start in range(0, len(texts), chunk_size):
        stop = min(start + chunk_size, len(texts))
        batch = list(texts[start:stop])
        batch_proba = model.predict_proba(batch)
        batch_pred = classes[np.argmax(batch_proba, axis=1)]
        gold_columns = np.array([class_to_column[int(label)] for label in labels_array[start:stop]])
        predictions.append(batch_pred)
        probabilities.append(batch_proba[np.arange(stop - start), gold_columns])
    return np.concatenate(predictions), np.concatenate(probabilities)

PREDICTION_COLUMNS = [
    'dataset', 'example_id', 'model', 'model_seed', 'attack', 'strength', 'attack_seed',
    'defense', 'gold', 'prediction', 'gold_probability', 'correct', 'changed_count',
]

def prediction_partition_path(source, dataset, model, model_seed, defense):
    return PATHS['predictions'] / source / dataset / model / f'seed_{model_seed}' / f'{defense}.parquet'

def write_prediction_partition(frame, source, dataset, model, model_seed, defense):
    path = prediction_partition_path(source, dataset, model, model_seed, defense)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame[PREDICTION_COLUMNS].to_parquet(path, index=False)
    return path


## Representations and fixed classifier

Word TF-IDF uses word unigrams and bigrams; character TF-IDF uses 3–5 character n-grams; BPE TF-IDF uses o200k token unigrams and bigrams. All three feed Logistic Regression with identical optimization settings. No accent stripping or Unicode normalization is applied during training.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

data = pd.read_parquet(PATHS['processed_data'])
attacks = pd.read_parquet(PATHS['attacks'])
model_seed = int(CONFIG['project']['root_seed'])

def vectorizer_for(model_name):
    common = dict(dtype=np.float32, min_df=2, sublinear_tf=True, strip_accents=None)
    if model_name == 'word_tfidf':
        return TfidfVectorizer(ngram_range=(1, 2), lowercase=True, **common)
    if model_name == 'char_tfidf':
        return TfidfVectorizer(analyzer='char', ngram_range=(3, 5), lowercase=True, **common)
    if model_name == 'bpe_tfidf':
        return TfidfVectorizer(tokenizer=bpe_tokenize, preprocessor=None, token_pattern=None, lowercase=False, ngram_range=(1, 2), **common)
    raise ValueError(model_name)

def build_model(model_name, c_value):
    return Pipeline([
        ('tfidf', vectorizer_for(model_name)),
        ('classifier', LogisticRegression(C=c_value, solver='lbfgs', max_iter=CONFIG['classical']['max_iter'], random_state=model_seed)),
    ])

def condition_frame(dataset_name):
    clean = data[(data['dataset'] == dataset_name) & (data['split'] == 'test')][['dataset', 'example_id', 'text', 'label']].copy()
    clean = clean.assign(attack='clean', strength=0.0, attack_seed=-1, changed_count=0)
    attacked = attacks[attacks['dataset'] == dataset_name][['dataset', 'example_id', 'attacked_text', 'label', 'attack', 'strength', 'attack_seed', 'changed_count']].rename(columns={'attacked_text': 'text'})
    return pd.concat([clean, attacked], ignore_index=True)


## Select regularization and predict every condition

For each dataset and representation, the configured values of `C` are compared using validation Macro-F1. Ties prefer the smaller `C`, after which the selected fitted model predicts the clean test set and every attacked condition. Gold-class probability is saved for confidence-drop and qualitative analyses.

In [ ]:
selection_rows = []
quick_rows = []
for dataset_name in ['banking77', 'trec']:
    train = data[(data['dataset'] == dataset_name) & (data['split'] == 'train')]
    validation = data[(data['dataset'] == dataset_name) & (data['split'] == 'validation')]
    conditions = condition_frame(dataset_name)
    for model_name in ['word_tfidf', 'char_tfidf', 'bpe_tfidf']:
        candidates = []
        for c_value in CONFIG['classical']['c_grid']:
            candidate = build_model(model_name, float(c_value))
            candidate.fit(train['text'], train['label'])
            prediction = candidate.predict(validation['text'])
            score = f1_score(validation['label'], prediction, average='macro', zero_division=0)
            candidates.append((score, float(c_value), candidate))
            selection_rows.append({'dataset': dataset_name, 'model': model_name, 'c': float(c_value), 'validation_macro_f1': score})
        best_score, best_c, model = max(candidates, key=lambda item: (item[0], -item[1]))
        model_path = PATHS['models'] / 'classical' / dataset_name / f'{model_name}.joblib'
        model_path.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(model, model_path)
        pred, gold_probability = predict_sklearn(model, conditions['text'].tolist(), conditions['label'].to_numpy())
        prediction_frame = conditions.drop(columns='text').rename(columns={'label': 'gold'}).assign(
            model=model_name, model_seed=model_seed, defense='none',
            prediction=pred.astype(int), gold_probability=gold_probability.astype(float),
        )
        prediction_frame['correct'] = prediction_frame['gold'].eq(prediction_frame['prediction'])
        path = write_prediction_partition(prediction_frame, 'raw', dataset_name, model_name, model_seed, 'none')
        clean_rows = prediction_frame[prediction_frame['attack'] == 'clean']
        quick_rows.append({'dataset': dataset_name, 'model': model_name, 'best_c': best_c, 'validation_macro_f1': best_score, 'clean_accuracy': accuracy_score(clean_rows['gold'], clean_rows['prediction']), 'clean_macro_f1': f1_score(clean_rows['gold'], clean_rows['prediction'], average='macro', zero_division=0), 'artifact': str(path.relative_to(ROOT))})

selection = pd.DataFrame(selection_rows)
selection.to_csv(PATHS['tables'] / 'classical_hyperparameter_search.csv', index=False)
quick = pd.DataFrame(quick_rows)
quick.to_csv(PATHS['tables'] / 'classical_clean_results.csv', index=False)
quick


## Interpretation

Clean accuracy and Macro-F1 confirm that each representation learned the task, while notebooks 06 and 08 quantify robustness. Because classifier family, split, tuning grid, and prediction conditions are shared, differences among these three systems are primarily attributable to token representation.